# Notebook 10: Reproducibility and Statistical Validation

**Author:** Anthony Amit Biswas

## What this notebook does

Produces confidence intervals, statistical comparisons, manifests, checksums, documentation, and a reproducibility package from frozen results.


## 1. Imports and Google Drive

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import textwrap
import warnings
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from scipy.stats import chi2, norm
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNNING_IN_COLAB = True
except Exception:
    RUNNING_IN_COLAB = False

print({
    "running_in_colab": RUNNING_IN_COLAB,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy_available": SCIPY_AVAILABLE,
})

## 2. Project configuration

Change `PROJECT_ROOT` only when the dissertation folder is stored elsewhere.

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

OUTPUTS_ROOT = PROJECT_ROOT / "outputs"

NOTEBOOK_07_DIR = OUTPUTS_ROOT / "notebook_07_final_evaluation"
NOTEBOOK_08_DIR = OUTPUTS_ROOT / "notebook_08_error_analysis"
NOTEBOOK_09_DIR = OUTPUTS_ROOT / "notebook_09_publication_assets"

NOTEBOOK_10_DIR = OUTPUTS_ROOT / "notebook_10_reproducibility_statistics_submission"

STATISTICS_DIR = NOTEBOOK_10_DIR / "statistics"
TABLES_DIR = NOTEBOOK_10_DIR / "tables"
REPORTS_DIR = NOTEBOOK_10_DIR / "reports"
MANIFESTS_DIR = NOTEBOOK_10_DIR / "manifests"
SUPPLEMENTARY_DIR = NOTEBOOK_10_DIR / "supplementary"
PACKAGE_DIR = NOTEBOOK_10_DIR / "submission_package"

DISSERTATION_PACKAGE_DIR = PACKAGE_DIR / "Dissertation"
POSTER_PACKAGE_DIR = PACKAGE_DIR / "Poster"
PRESENTATION_PACKAGE_DIR = PACKAGE_DIR / "Presentation"
GITHUB_PACKAGE_DIR = PACKAGE_DIR / "GitHub"
SUPPLEMENTARY_PACKAGE_DIR = PACKAGE_DIR / "Supplementary"
STATISTICS_PACKAGE_DIR = PACKAGE_DIR / "Statistics"

for directory in [
    NOTEBOOK_10_DIR, STATISTICS_DIR, TABLES_DIR, REPORTS_DIR, MANIFESTS_DIR,
    SUPPLEMENTARY_DIR, PACKAGE_DIR, DISSERTATION_PACKAGE_DIR,
    POSTER_PACKAGE_DIR, PRESENTATION_PACKAGE_DIR, GITHUB_PACKAGE_DIR,
    SUPPLEMENTARY_PACKAGE_DIR, STATISTICS_PACKAGE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260729
BOOTSTRAP_ITERATIONS = 10_000
CONFIDENCE_LEVEL = 0.95

rng = np.random.default_rng(RANDOM_SEED)

print("Notebook 10 output:", NOTEBOOK_10_DIR)

## 3. File-discovery helpers

The earlier notebooks may have been stored with small naming differences. These helpers search predictable locations without modifying source outputs.

In [ ]:
def find_file(root: Path, filename: str) -> Path | None:
    if not root.exists():
        return None
    direct = root / filename
    if direct.exists():
        return direct
    matches = list(root.rglob(filename))
    return matches[0] if matches else None


def find_first(root: Path, candidates: list[str]) -> Path | None:
    for filename in candidates:
        path = find_file(root, filename)
        if path is not None:
            return path
    return None


def load_csv(path: Path | None, required: bool = True) -> pd.DataFrame:
    if path is None or not path.exists():
        if required:
            raise FileNotFoundError(str(path))
        return pd.DataFrame()
    return pd.read_csv(path)


def first_column(df: pd.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    lower_map = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    if required:
        raise KeyError(f"Expected one of {candidates}; found {list(df.columns)}")
    return None


def safe_copy(source: Path | None, destination: Path) -> bool:
    if source is None or not source.exists():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(source, destination)
    return True


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def relative_to_project(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

## 4. Validate frozen upstream outputs

In [ ]:
upstream_checks = []

for notebook_name, directory, validation_candidates in [
    (
        "Notebook 08",
        NOTEBOOK_08_DIR,
        ["notebook_08_final_validation.csv"],
    ),
    (
        "Notebook 09",
        NOTEBOOK_09_DIR,
        ["notebook_09_final_validation.csv"],
    ),
]:
    validation_path = find_first(directory / "reports", validation_candidates)
    exists = validation_path is not None
    passed = False
    details = "validation report not found"

    if exists:
        validation_df = pd.read_csv(validation_path)
        pass_col = first_column(validation_df, ["Passed", "passed"], required=False)
        if pass_col is not None:
            passed = validation_df[pass_col].astype(bool).all()
            details = f"{len(validation_df)} checks; all_passed={passed}"
        else:
            details = "validation report has no Passed column"

    upstream_checks.append({
        "component": notebook_name,
        "directory_exists": directory.exists(),
        "validation_report": str(validation_path) if validation_path else "",
        "passed": bool(passed),
        "details": details,
    })

upstream_validation_df = pd.DataFrame(upstream_checks)
display(upstream_validation_df)

if not upstream_validation_df["passed"].all():
    raise RuntimeError(
        "Notebook 08 or Notebook 09 validation has not passed. "
        "Resolve upstream validation before creating the final package."
    )

## 5. Load frozen evaluation results

In [ ]:
summary_path = find_first(
    NOTEBOOK_07_DIR,
    [
        "notebook_07_final_evaluation_summary.csv",
        "final_evaluation_summary.csv",
    ],
)

if summary_path is None:
    summary_path = find_first(
        PROJECT_ROOT,
        ["notebook_07_final_evaluation_summary.csv"],
    )

evaluation_summary_df = load_csv(summary_path, required=True)
display(evaluation_summary_df)

overall_results_path = find_first(
    NOTEBOOK_09_DIR / "tables",
    ["table_04_dissertation_overall_results.csv"],
)

overall_results_df = load_csv(overall_results_path, required=True)
display(overall_results_df)

## 6. Software versions and environment specification

In [ ]:
packages_to_record = [
    "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
    "seaborn", "spacy", "transformers", "torch", "tensorflow",
    "nltk", "statsmodels", "jupyter", "ipykernel", "nbformat",
]

version_rows = []

for package_name in packages_to_record:
    try:
        version = importlib.metadata.version(package_name)
        status = "installed"
    except importlib.metadata.PackageNotFoundError:
        version = ""
        status = "not installed"

    version_rows.append({
        "package": package_name,
        "version": version,
        "status": status,
    })

software_versions_df = pd.DataFrame(version_rows)

system_rows = [
    {"component": "Python", "version": platform.python_version()},
    {"component": "Platform", "version": platform.platform()},
    {"component": "Processor", "version": platform.processor()},
    {"component": "Machine", "version": platform.machine()},
]

software_versions_df.to_csv(
    MANIFESTS_DIR / "software_versions.csv",
    index=False,
)

pd.DataFrame(system_rows).to_csv(
    MANIFESTS_DIR / "system_environment.csv",
    index=False,
)

display(software_versions_df)

In [ ]:
requirements_lines = [
    f"{row.package}=={row.version}"
    for row in software_versions_df.itertuples()
    if row.status == "installed" and row.version
]

requirements_text = "\n".join(requirements_lines) + "\n"
(GITHUB_PACKAGE_DIR / "requirements.txt").write_text(
    requirements_text,
    encoding="utf-8",
)

environment_lines = [
    "name: clinical-nlp-dissertation",
    "channels:",
    "  - conda-forge",
    "dependencies:",
    f"  - python={platform.python_version()}",
    "  - pip",
    "  - pip:",
]
environment_lines.extend([f"      - {line}" for line in requirements_lines])

(GITHUB_PACKAGE_DIR / "environment.yml").write_text(
    "\n".join(environment_lines) + "\n",
    encoding="utf-8",
)

print("Environment files generated.")

# Statistical validation

The preferred unit for uncertainty estimation is the **discharge summary**, because mentions within the same note are not independent. The bootstrap therefore resamples notes with replacement whenever note identifiers are available.

A McNemar test is only valid for paired binary outcomes from the same observational units. The notebook performs it only when it can construct such a paired table; otherwise it records the test as not applicable rather than manufacturing a comparison.

## 7. Locate evaluation detail files

In [ ]:
detail_files = {
    "exact_matches": find_first(PROJECT_ROOT, ["exact_span_matches.csv"]),
    "exact_fp": find_first(PROJECT_ROOT, ["exact_span_false_positives.csv"]),
    "exact_fn": find_first(PROJECT_ROOT, ["exact_span_false_negatives.csv"]),
    "relaxed_category_matches": find_first(PROJECT_ROOT, ["relaxed_overlap_category_matches.csv"]),
    "strict_detail": find_first(PROJECT_ROOT, ["strict_exact_match_attribute_detail.csv"]),
    "category_metrics": find_first(
        NOTEBOOK_08_DIR / "tables",
        ["table_09_per_category_exact_span_category_metrics.csv"],
    ),
}

pd.DataFrame([
    {"dataset": key, "path": str(value) if value else "", "found": value is not None}
    for key, value in detail_files.items()
])

## 8. Metric and interval functions

In [ ]:
def precision_recall_f1(tp: float, fp: float, fn: float) -> tuple[float, float, float]:
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if np.isfinite(precision) and np.isfinite(recall) and (precision + recall)
        else np.nan
    )
    return precision, recall, f1


def wilson_interval(successes: int, total: int, confidence: float = 0.95) -> tuple[float, float]:
    if total <= 0:
        return np.nan, np.nan

    alpha = 1 - confidence
    z = norm.ppf(1 - alpha / 2) if SCIPY_AVAILABLE else 1.959963984540054

    phat = successes / total
    denominator = 1 + z**2 / total
    centre = (phat + z**2 / (2 * total)) / denominator
    half_width = (
        z
        * math.sqrt(phat * (1 - phat) / total + z**2 / (4 * total**2))
        / denominator
    )
    return max(0.0, centre - half_width), min(1.0, centre + half_width)


def percentile_interval(values: np.ndarray, confidence: float = 0.95) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan

    alpha = 1 - confidence
    return (
        float(np.quantile(values, alpha / 2)),
        float(np.quantile(values, 1 - alpha / 2)),
    )


def cohen_h(p1: float, p2: float) -> float:
    if not (0 <= p1 <= 1 and 0 <= p2 <= 1):
        return np.nan
    return 2 * math.asin(math.sqrt(p1)) - 2 * math.asin(math.sqrt(p2))

## 9. Build note-level exact-span counts

In [ ]:
def identify_note_column(df: pd.DataFrame) -> str | None:
    return first_column(
        df,
        [
            "note_id", "NOTE_ID", "note", "Note ID",
            "subject_note_id", "registered_note_id",
        ],
        required=False,
    )


exact_matches_df = load_csv(detail_files["exact_matches"], required=False)
exact_fp_df = load_csv(detail_files["exact_fp"], required=False)
exact_fn_df = load_csv(detail_files["exact_fn"], required=False)

note_level_exact_df = pd.DataFrame()

if not exact_matches_df.empty and not exact_fp_df.empty and not exact_fn_df.empty:
    match_note_col = identify_note_column(exact_matches_df)
    fp_note_col = identify_note_column(exact_fp_df)
    fn_note_col = identify_note_column(exact_fn_df)

    if all([match_note_col, fp_note_col, fn_note_col]):
        tp_counts = exact_matches_df.groupby(match_note_col).size().rename("tp")
        fp_counts = exact_fp_df.groupby(fp_note_col).size().rename("fp")
        fn_counts = exact_fn_df.groupby(fn_note_col).size().rename("fn")

        note_level_exact_df = pd.concat(
            [tp_counts, fp_counts, fn_counts],
            axis=1,
        ).fillna(0).reset_index()

        note_level_exact_df = note_level_exact_df.rename(
            columns={note_level_exact_df.columns[0]: "note_id"}
        )

        for col in ["tp", "fp", "fn"]:
            note_level_exact_df[col] = note_level_exact_df[col].astype(int)

if note_level_exact_df.empty:
    print(
        "Note-level exact-span files or note identifiers were not available. "
        "Bootstrap confidence intervals will be marked unavailable rather than "
        "using an invalid mention-level independence assumption."
    )
else:
    display(note_level_exact_df.head())
    print("Notes represented:", note_level_exact_df["note_id"].nunique())

## 10. Note-level bootstrap confidence intervals

In [ ]:
bootstrap_rows = []

if not note_level_exact_df.empty:
    note_counts = note_level_exact_df[["tp", "fp", "fn"]].to_numpy(dtype=int)
    n_notes = len(note_counts)

    bootstrap_metrics = np.empty((BOOTSTRAP_ITERATIONS, 3), dtype=float)

    for iteration in range(BOOTSTRAP_ITERATIONS):
        indices = rng.integers(0, n_notes, size=n_notes)
        sampled = note_counts[indices].sum(axis=0)
        bootstrap_metrics[iteration] = precision_recall_f1(
            tp=sampled[0],
            fp=sampled[1],
            fn=sampled[2],
        )

    observed = precision_recall_f1(
        tp=note_counts[:, 0].sum(),
        fp=note_counts[:, 1].sum(),
        fn=note_counts[:, 2].sum(),
    )

    for metric_index, metric_name in enumerate(["Precision", "Recall", "F1"]):
        lower, upper = percentile_interval(
            bootstrap_metrics[:, metric_index],
            CONFIDENCE_LEVEL,
        )
        bootstrap_rows.append({
            "evaluation": "Exact span",
            "metric": metric_name,
            "estimate": observed[metric_index],
            "ci_lower": lower,
            "ci_upper": upper,
            "confidence_level": CONFIDENCE_LEVEL,
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
            "resampling_unit": "discharge summary",
            "status": "estimated",
        })
else:
    for metric_name in ["Precision", "Recall", "F1"]:
        bootstrap_rows.append({
            "evaluation": "Exact span",
            "metric": metric_name,
            "estimate": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "confidence_level": CONFIDENCE_LEVEL,
            "bootstrap_iterations": 0,
            "resampling_unit": "discharge summary",
            "status": "not estimated: note-level counts unavailable",
        })

bootstrap_ci_df = pd.DataFrame(bootstrap_rows)
bootstrap_ci_df.to_csv(
    STATISTICS_DIR / "bootstrap_confidence_intervals.csv",
    index=False,
)
display(bootstrap_ci_df)

## 11. Wilson confidence intervals for accuracy-type measures

In [ ]:
wilson_rows = []

# Category accuracy from exact-span category classification, if available.
category_classification_path = find_first(
    PROJECT_ROOT,
    ["exact_span_category_classification.csv"],
)
category_classification_df = load_csv(category_classification_path, required=False)

if not category_classification_df.empty:
    correct_col = first_column(
        category_classification_df,
        ["correct", "Correct", "is_correct", "category_correct"],
        required=False,
    )
    gold_col = first_column(
        category_classification_df,
        ["gold_category", "Gold Category", "gold_label"],
        required=False,
    )
    pred_col = first_column(
        category_classification_df,
        ["predicted_category", "Predicted Category", "predicted_label"],
        required=False,
    )

    if correct_col is not None:
        correct = category_classification_df[correct_col].astype(bool)
    elif gold_col is not None and pred_col is not None:
        correct = (
            category_classification_df[gold_col].astype(str)
            == category_classification_df[pred_col].astype(str)
        )
    else:
        correct = pd.Series(dtype=bool)

    if len(correct):
        successes = int(correct.sum())
        total = int(len(correct))
        lower, upper = wilson_interval(successes, total, CONFIDENCE_LEVEL)
        wilson_rows.append({
            "measure": "Exact-span category accuracy",
            "successes": successes,
            "total": total,
            "estimate": successes / total,
            "ci_lower": lower,
            "ci_upper": upper,
            "confidence_level": CONFIDENCE_LEVEL,
        })

for label, filename in [
    ("Assertion accuracy", "assertion_evaluation_exact_spans.csv"),
    ("Temporality accuracy", "temporality_evaluation_exact_spans.csv"),
]:
    path = find_first(PROJECT_ROOT, [filename])
    df = load_csv(path, required=False)
    if df.empty:
        continue

    correct_col = first_column(
        df,
        ["correct", "Correct", "is_correct", "attribute_correct"],
        required=False,
    )
    gold_col = first_column(
        df,
        ["gold_label", "Gold", "gold"],
        required=False,
    )
    pred_col = first_column(
        df,
        ["predicted_label", "Predicted", "prediction"],
        required=False,
    )

    if correct_col is not None:
        correct = df[correct_col].astype(bool)
    elif gold_col is not None and pred_col is not None:
        correct = df[gold_col].astype(str) == df[pred_col].astype(str)
    else:
        continue

    successes = int(correct.sum())
    total = int(len(correct))
    lower, upper = wilson_interval(successes, total, CONFIDENCE_LEVEL)

    wilson_rows.append({
        "measure": label,
        "successes": successes,
        "total": total,
        "estimate": successes / total,
        "ci_lower": lower,
        "ci_upper": upper,
        "confidence_level": CONFIDENCE_LEVEL,
    })

wilson_ci_df = pd.DataFrame(wilson_rows)
wilson_ci_df.to_csv(
    STATISTICS_DIR / "wilson_confidence_intervals.csv",
    index=False,
)
display(wilson_ci_df)

## 12. Per-category confidence intervals

Precision and recall receive Wilson intervals because each is a binomial proportion conditional on predicted or gold support. F1 is reported with a conservative interval derived from the lower and upper precision/recall bounds; this is descriptive and is labelled accordingly.

In [ ]:
category_metrics_df = load_csv(detail_files["category_metrics"], required=True)

cat_col = first_column(category_metrics_df, ["Category", "category"])
tp_col = first_column(
    category_metrics_df,
    ["TP", "tp", "True Positives", "true positives", "true_positives"],
    required=False,
)
fp_col = first_column(
    category_metrics_df,
    ["FP", "fp", "False Positives", "false positives", "false_positives"],
    required=False,
)
fn_col = first_column(
    category_metrics_df,
    ["FN", "fn", "False Negatives", "false negatives", "false_negatives"],
    required=False,
)

per_category_ci_rows = []

if all([tp_col, fp_col, fn_col]):
    for _, record in category_metrics_df.iterrows():
        category = record[cat_col]
        tp = int(record[tp_col])
        fp = int(record[fp_col])
        fn = int(record[fn_col])

        p, r, f1 = precision_recall_f1(tp, fp, fn)
        p_low, p_high = wilson_interval(tp, tp + fp, CONFIDENCE_LEVEL)
        r_low, r_high = wilson_interval(tp, tp + fn, CONFIDENCE_LEVEL)

        f1_low = (
            2 * p_low * r_low / (p_low + r_low)
            if (p_low + r_low) > 0 else 0.0
        )
        f1_high = (
            2 * p_high * r_high / (p_high + r_high)
            if (p_high + r_high) > 0 else 0.0
        )

        per_category_ci_rows.append({
            "category": category,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": p,
            "precision_ci_lower": p_low,
            "precision_ci_upper": p_high,
            "recall": r,
            "recall_ci_lower": r_low,
            "recall_ci_upper": r_high,
            "f1": f1,
            "f1_descriptive_lower": f1_low,
            "f1_descriptive_upper": f1_high,
            "confidence_level": CONFIDENCE_LEVEL,
            "f1_interval_note": "derived from Wilson precision/recall bounds; not a bootstrap CI",
        })
else:
    print("Per-category TP/FP/FN columns were not available.")

per_category_ci_df = pd.DataFrame(per_category_ci_rows)
per_category_ci_df.to_csv(
    STATISTICS_DIR / "per_category_confidence_intervals.csv",
    index=False,
)

if per_category_ci_df.empty:
    raise RuntimeError(
        "Per-category confidence intervals were not generated. "
        "Check the TP, FP and FN column names in the source category table."
    )

if len(per_category_ci_df) != len(category_metrics_df):
    raise RuntimeError(
        f"Expected {len(category_metrics_df)} category rows but generated "
        f"{len(per_category_ci_df)}."
    )

display(per_category_ci_df.head(10))

## 13. Paired McNemar test

The intended comparison is exact-span category success versus relaxed-overlap category success for the same gold mention. This tests whether relaxed matching recovers significantly more gold mentions. The test is run only if stable gold-mention identifiers can be aligned one-to-one.

In [ ]:
def exact_mcnemar(b: int, c: int) -> tuple[float, float]:
    # Exact two-sided binomial McNemar p-value.
    n = b + c
    if n == 0:
        return 0.0, 1.0

    k = min(b, c)
    probability = sum(
        math.comb(n, i) * (0.5 ** n)
        for i in range(k + 1)
    )
    return float(n), min(1.0, 2 * probability)


mcnemar_result = {
    "comparison": "Exact-span category success vs relaxed-overlap category success",
    "paired_unit": "gold mention",
    "b_exact_success_relaxed_failure": np.nan,
    "c_exact_failure_relaxed_success": np.nan,
    "discordant_pairs": np.nan,
    "exact_two_sided_p_value": np.nan,
    "status": "not performed",
    "reason": "",
}

exact_cat_path = find_first(PROJECT_ROOT, ["exact_span_category_classification.csv"])
relaxed_cat_path = detail_files["relaxed_category_matches"]

exact_cat_df = load_csv(exact_cat_path, required=False)
relaxed_cat_df = load_csv(relaxed_cat_path, required=False)

if not exact_cat_df.empty and not relaxed_cat_df.empty:
    id_candidates = [
        "gold_mention_id", "gold_id", "mention_id",
        "annotation_id", "gold_span_id",
    ]

    exact_id_col = first_column(exact_cat_df, id_candidates, required=False)
    relaxed_id_col = first_column(relaxed_cat_df, id_candidates, required=False)

    if exact_id_col is not None and relaxed_id_col is not None:
        exact_success_col = first_column(
            exact_cat_df,
            ["correct", "category_correct", "is_correct"],
            required=False,
        )
        relaxed_success_col = first_column(
            relaxed_cat_df,
            ["correct", "category_correct", "is_correct"],
            required=False,
        )

        if exact_success_col is None:
            exact_gold_col = first_column(
                exact_cat_df, ["gold_category", "gold_label"], required=False
            )
            exact_pred_col = first_column(
                exact_cat_df, ["predicted_category", "predicted_label"], required=False
            )
            if exact_gold_col and exact_pred_col:
                exact_cat_df["_success"] = (
                    exact_cat_df[exact_gold_col].astype(str)
                    == exact_cat_df[exact_pred_col].astype(str)
                )
                exact_success_col = "_success"

        if relaxed_success_col is None:
            relaxed_gold_col = first_column(
                relaxed_cat_df, ["gold_category", "gold_label"], required=False
            )
            relaxed_pred_col = first_column(
                relaxed_cat_df, ["predicted_category", "predicted_label"], required=False
            )
            if relaxed_gold_col and relaxed_pred_col:
                relaxed_cat_df["_success"] = (
                    relaxed_cat_df[relaxed_gold_col].astype(str)
                    == relaxed_cat_df[relaxed_pred_col].astype(str)
                )
                relaxed_success_col = "_success"

        if exact_success_col and relaxed_success_col:
            paired = (
                exact_cat_df[[exact_id_col, exact_success_col]]
                .rename(columns={
                    exact_id_col: "gold_mention_id",
                    exact_success_col: "exact_success",
                })
                .merge(
                    relaxed_cat_df[[relaxed_id_col, relaxed_success_col]]
                    .rename(columns={
                        relaxed_id_col: "gold_mention_id",
                        relaxed_success_col: "relaxed_success",
                    }),
                    on="gold_mention_id",
                    how="inner",
                    validate="one_to_one",
                )
            )

            paired["exact_success"] = paired["exact_success"].astype(bool)
            paired["relaxed_success"] = paired["relaxed_success"].astype(bool)

            b = int((paired["exact_success"] & ~paired["relaxed_success"]).sum())
            c = int((~paired["exact_success"] & paired["relaxed_success"]).sum())
            discordant, p_value = exact_mcnemar(b, c)

            mcnemar_result.update({
                "b_exact_success_relaxed_failure": b,
                "c_exact_failure_relaxed_success": c,
                "discordant_pairs": int(discordant),
                "exact_two_sided_p_value": p_value,
                "status": "performed",
                "reason": f"{len(paired)} gold mentions aligned one-to-one",
            })
        else:
            mcnemar_result["reason"] = "success indicators could not be derived"
    else:
        mcnemar_result["reason"] = "stable gold-mention identifiers unavailable"
else:
    mcnemar_result["reason"] = "required exact and relaxed detail files unavailable"

mcnemar_df = pd.DataFrame([mcnemar_result])
mcnemar_df.to_csv(
    STATISTICS_DIR / "mcnemar_test.csv",
    index=False,
)
display(mcnemar_df)

## 14. Effect-size summary

In [ ]:
effect_rows = []

result_lookup = {
    str(row["Evaluation level"]): row
    for _, row in overall_results_df.iterrows()
}

comparisons = [
    ("Exact span", "Exact span + category"),
    ("Exact span + category", "Strict full extraction"),
    ("Exact span + category", "Relaxed overlap + category"),
]

for baseline, comparison in comparisons:
    if baseline not in result_lookup or comparison not in result_lookup:
        continue

    baseline_f1 = float(result_lookup[baseline]["F1-score"])
    comparison_f1 = float(result_lookup[comparison]["F1-score"])

    absolute_change = comparison_f1 - baseline_f1
    relative_change = (
        absolute_change / baseline_f1
        if baseline_f1 != 0 else np.nan
    )

    effect_rows.append({
        "baseline": baseline,
        "comparison": comparison,
        "baseline_f1": baseline_f1,
        "comparison_f1": comparison_f1,
        "absolute_change": absolute_change,
        "relative_change": relative_change,
        "cohen_h_descriptive": cohen_h(comparison_f1, baseline_f1),
        "interpretation_note": (
            "Cohen's h is descriptive here because F1 is not a simple binomial proportion."
        ),
    })

effect_size_df = pd.DataFrame(effect_rows)
effect_size_df.to_csv(
    STATISTICS_DIR / "effect_size_summary.csv",
    index=False,
)
display(effect_size_df)

# Dissertation and appendix materials

## 15. Formatted dissertation tables

In [ ]:
def format_ci(estimate, lower, upper, decimals=3):
    if not all(np.isfinite([estimate, lower, upper])):
        return "Not estimated"
    return f"{estimate:.{decimals}f} ({lower:.{decimals}f}–{upper:.{decimals}f})"


formatted_overall_df = overall_results_df.copy()
for column in ["Precision", "Recall", "F1-score"]:
    if column in formatted_overall_df.columns:
        formatted_overall_df[column] = formatted_overall_df[column].map(
            lambda value: f"{float(value):.4f}"
        )

formatted_overall_df.to_csv(
    TABLES_DIR / "dissertation_table_overall_performance.csv",
    index=False,
)

if not bootstrap_ci_df.empty:
    formatted_bootstrap_df = bootstrap_ci_df.copy()
    formatted_bootstrap_df["estimate_95_ci"] = formatted_bootstrap_df.apply(
        lambda row: format_ci(
            row["estimate"], row["ci_lower"], row["ci_upper"]
        ),
        axis=1,
    )
    formatted_bootstrap_df.to_csv(
        TABLES_DIR / "dissertation_table_bootstrap_intervals.csv",
        index=False,
    )

display(formatted_overall_df)

## 16. Appendix table selection

In [ ]:
appendix_sources = {
    "appendix_per_category_metrics.csv": detail_files["category_metrics"],
    "appendix_per_category_confidence_intervals.csv": (
        STATISTICS_DIR / "per_category_confidence_intervals.csv"
    ),
    "appendix_assertion_confusion_matrix.csv": find_first(
        PROJECT_ROOT, ["assertion_confusion_matrix.csv"]
    ),
    "appendix_temporality_confusion_matrix.csv": find_first(
        PROJECT_ROOT, ["temporality_confusion_matrix.csv"]
    ),
    "appendix_false_negative_root_causes.csv": find_first(
        PROJECT_ROOT, ["false_negative_root_cause_summary.csv"]
    ),
    "appendix_lexicon_gap_expressions.csv": find_first(
        PROJECT_ROOT, ["lexicon_gap_expressions.csv"]
    ),
}

appendix_inventory = []

for destination_name, source in appendix_sources.items():
    destination = SUPPLEMENTARY_DIR / destination_name
    copied = safe_copy(source, destination)
    appendix_inventory.append({
        "source": str(source) if source else "",
        "destination": str(destination),
        "copied": copied,
    })

appendix_inventory_df = pd.DataFrame(appendix_inventory)
appendix_inventory_df.to_csv(
    MANIFESTS_DIR / "appendix_inventory.csv",
    index=False,
)
display(appendix_inventory_df)

## 17. Figure numbering and caption register

In [ ]:
asset_manifest_path = find_first(
    NOTEBOOK_09_DIR / "tables",
    ["table_03_final_asset_manifest.csv"],
)
asset_manifest_df = load_csv(asset_manifest_path, required=True)

figure_col = first_column(asset_manifest_df, ["Figure"])
section_col = first_column(
    asset_manifest_df,
    ["Dissertation section"],
    required=False,
)

caption_map = {
    "figure_01_evaluation_performance_ladder":
        "Precision, recall and F1-score under increasingly strict evaluation criteria.",
    "figure_02_contextual_attribute_penalty":
        "Reduction in F1-score when category and contextual attributes are required.",
    "figure_03_poster_metric_cards":
        "Corpus size and headline evaluation results.",
    "figure_04_span_error_profile_by_category":
        "False-positive and false-negative counts for the categories contributing most span-level errors.",
    "figure_05_false_negative_root_causes":
        "Root-cause classification of false-negative mentions.",
    "figure_06_category_f1_with_support":
        "Per-category exact-span classification F1-score with gold-standard support.",
    "figure_07_focused_category_error_pattern":
        "Observed exact-span category confusion between pneumonia and aspiration.",
    "figure_08_assertion_error_profile":
        "Row-normalised assertion misclassification profile.",
    "figure_09_temporality_error_profile":
        "Row-normalised temporality misclassification profile.",
    "figure_10_context_attribute_summary":
        "Accuracy, support-aware macro F1 and weighted F1 for assertion and temporality.",
    "figure_11_primary_error_taxonomy":
        "Mutually exclusive primary error taxonomy.",
    "figure_12_key_findings_panel":
        "Summary of the principal evaluation findings.",
}

figure_register_rows = []

for index, row in asset_manifest_df.iterrows():
    stem = str(row[figure_col])
    figure_register_rows.append({
        "figure_number": index + 1,
        "file_stem": stem,
        "caption": caption_map.get(stem, ""),
        "recommended_section": row[section_col] if section_col else "",
    })

figure_register_df = pd.DataFrame(figure_register_rows)
figure_register_df.to_csv(
    TABLES_DIR / "figure_numbering_and_captions.csv",
    index=False,
)
display(figure_register_df)

# Reproducibility manifests

## 18. Experiment manifest

In [ ]:
def metric_from_summary(label_fragments: list[str]) -> dict:
    text_cols = [
        c for c in evaluation_summary_df.columns
        if evaluation_summary_df[c].dtype == "object"
    ]
    for _, row in evaluation_summary_df.iterrows():
        joined = " ".join(str(row[c]) for c in text_cols).lower()
        if all(fragment.lower() in joined for fragment in label_fragments):
            return row.to_dict()
    return {}


experiment_manifest = {
    "project_title":
        "Automatic Detection of Clinical Complications from ICU Discharge Summaries Using Clinical NLP",
    "author": "Anthony Amit Biswas",
    "dataset": "MIMIC-IV discharge summaries",
    "reviewed_notes": 70,
    "gold_mentions": 491,
    "canonical_predictions": 546,
    "random_seed": RANDOM_SEED,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "confidence_level": CONFIDENCE_LEVEL,
    "evaluation_summary_file": relative_to_project(summary_path),
    "notebook_08_validation": upstream_checks[0],
    "notebook_09_validation": upstream_checks[1],
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "research_constraints": {
        "gold_standard_modified": False,
        "predictions_modified": False,
        "evaluation_recomputed": False,
        "raw_patient_text_in_public_package": False,
    },
}

with (MANIFESTS_DIR / "experiment_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(experiment_manifest, handle, indent=2)

display(experiment_manifest)

## 19. Complete project inventory

In [ ]:
inventory_roots = [
    NOTEBOOK_07_DIR,
    NOTEBOOK_08_DIR,
    NOTEBOOK_09_DIR,
    NOTEBOOK_10_DIR,
]

inventory_rows = []

for root in inventory_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        stat = path.stat()
        inventory_rows.append({
            "path": relative_to_project(path),
            "notebook_stage": root.name,
            "extension": path.suffix.lower(),
            "size_bytes": stat.st_size,
            "modified_utc": datetime.fromtimestamp(
                stat.st_mtime, tz=timezone.utc
            ).isoformat(),
        })

project_inventory_df = pd.DataFrame(inventory_rows).sort_values("path")
project_inventory_df.to_csv(
    MANIFESTS_DIR / "project_inventory.csv",
    index=False,
)

print("Inventory files:", len(project_inventory_df))
display(project_inventory_df.head(20))

## 20. SHA256 integrity manifest

In [ ]:
hash_roots = [
    NOTEBOOK_07_DIR,
    NOTEBOOK_08_DIR,
    NOTEBOOK_09_DIR,
    TABLES_DIR,
    STATISTICS_DIR,
    REPORTS_DIR,
    MANIFESTS_DIR,
]

hash_rows = []

for root in hash_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.is_file() and path.name != "sha256_manifest.csv":
            hash_rows.append({
                "path": relative_to_project(path),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })

sha256_manifest_df = pd.DataFrame(hash_rows).sort_values("path")
sha256_manifest_df.to_csv(
    MANIFESTS_DIR / "sha256_manifest.csv",
    index=False,
)

print("Hashed files:", len(sha256_manifest_df))
display(sha256_manifest_df.head())

# GitHub documentation

## 21. README, citation and licence files

The README avoids publishing MIMIC-IV notes, annotations containing patient text or restricted source data.

In [ ]:
readme_text = '''# Clinical Complication Extraction from ICU Discharge Summaries

This repository contains a reproducible implementation and evaluation workflow for clinical complication detection from ICU discharge summaries.

## Research objective

The project evaluates a clinical NLP pipeline for detecting clinical complication mentions in ICU discharge summaries and assigning complication category, assertion and temporality attributes.

## Dataset access

The study uses MIMIC-IV data under the PhysioNet credentialing and data-use requirements. Raw clinical notes and any derived files containing restricted patient text are not distributed in this repository.

## Pipeline

The repository is organised as ten sequential notebooks covering data preparation, annotation, NLP extraction, canonicalisation, final evaluation, error analysis, publication-quality visualisation, statistical validation and reproducibility packaging.

## Headline evaluation

- Reviewed discharge summaries: 70
- Gold-standard complication mentions: 491
- Canonical system predictions: 546
- Exact-span F1: 0.9026
- Exact-span plus category F1: 0.8949
- Relaxed-overlap plus category F1: 0.9238
- Strict end-to-end F1: 0.5767

## Reproducibility

Notebook 10 records software versions, confidence intervals, project manifests, SHA256 checksums and a structured submission package.

## Privacy and licensing

No raw MIMIC-IV clinical text is included. Users must obtain their own authorised PhysioNet access and comply with the relevant data-use agreement.
'''

(GITHUB_PACKAGE_DIR / "README.md").write_text(readme_text, encoding="utf-8")

citation_text = '''cff-version: 1.2.0
message: "If you use this work, please cite it using the metadata below."
title: "Automatic Detection of Clinical Complications from ICU Discharge Summaries Using Clinical NLP"
type: software
authors:
  - family-names: "Biswas"
    given-names: "Anthony Amit"
date-released: "2026"
version: "1.0.0"
license: MIT
'''

(GITHUB_PACKAGE_DIR / "CITATION.cff").write_text(citation_text, encoding="utf-8")

license_text = '''MIT License

Copyright (c) 2026 Anthony Amit Biswas

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
'''

(GITHUB_PACKAGE_DIR / "LICENSE").write_text(license_text, encoding="utf-8")

print("GitHub documentation generated.")

In [ ]:
project_structure_text = '''# Project structure

```text
notebooks/
  01_...ipynb
  02_...ipynb
  ...
  10_reproducibility_statistics_and_submission_package.ipynb

outputs/
  notebook_07_final_evaluation/
  notebook_08_error_analysis/
  notebook_09_publication_assets/
  notebook_10_reproducibility_statistics_submission/

submission_package/
  Dissertation/
    Figures/
    Tables/
    Appendix/
  Poster/
  Presentation/
  GitHub/
  Supplementary/
  Statistics/
```

Restricted MIMIC-IV source data and clinical text are intentionally excluded from the public repository.
'''

(GITHUB_PACKAGE_DIR / "PROJECT_STRUCTURE.md").write_text(
    project_structure_text,
    encoding="utf-8",
)

changelog_text = '''# Changelog

## 1.0.0 — 2026

- Completed canonical prediction validation.
- Completed exact, relaxed and strict end-to-end evaluation.
- Added structured error analysis.
- Added publication-quality dissertation, poster and presentation figures.
- Added statistical uncertainty estimates and reproducibility manifests.
- Added final submission and GitHub packaging.
'''

(GITHUB_PACKAGE_DIR / "CHANGELOG.md").write_text(
    changelog_text,
    encoding="utf-8",
)

# Submission package

## 22. Copy dissertation assets

In [ ]:
# Clean package subdirectories before copying to prevent legacy files.
for directory in [
    DISSERTATION_PACKAGE_DIR,
    POSTER_PACKAGE_DIR,
    PRESENTATION_PACKAGE_DIR,
    SUPPLEMENTARY_PACKAGE_DIR,
    STATISTICS_PACKAGE_DIR,
]:
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)

safe_copy(
    NOTEBOOK_09_DIR / "dissertation_figures",
    DISSERTATION_PACKAGE_DIR / "Figures",
)
safe_copy(
    NOTEBOOK_09_DIR / "tables",
    DISSERTATION_PACKAGE_DIR / "Tables" / "Notebook_09",
)
safe_copy(
    TABLES_DIR,
    DISSERTATION_PACKAGE_DIR / "Tables" / "Notebook_10",
)
safe_copy(
    SUPPLEMENTARY_DIR,
    DISSERTATION_PACKAGE_DIR / "Appendix",
)

safe_copy(
    NOTEBOOK_09_DIR / "poster_figures",
    POSTER_PACKAGE_DIR / "Figures",
)
safe_copy(
    NOTEBOOK_09_DIR / "presentation_figures",
    PRESENTATION_PACKAGE_DIR / "Figures",
)
safe_copy(
    NOTEBOOK_09_DIR / "social_media_figures",
    GITHUB_PACKAGE_DIR / "assets",
)

safe_copy(SUPPLEMENTARY_DIR, SUPPLEMENTARY_PACKAGE_DIR)
safe_copy(STATISTICS_DIR, STATISTICS_PACKAGE_DIR)

print("Submission directories populated.")

## 23. Supplementary research report

In [ ]:
statistical_notes = []

if bootstrap_ci_df["status"].eq("estimated").any():
    statistical_notes.append(
        "Exact-span precision, recall and F1 uncertainty was estimated by "
        f"{BOOTSTRAP_ITERATIONS:,} note-level bootstrap resamples."
    )
else:
    statistical_notes.append(
        "Note-level bootstrap confidence intervals were not estimated because "
        "the required note identifiers were unavailable in the exported detail files."
    )

if mcnemar_result["status"] == "performed":
    statistical_notes.append(
        "An exact paired McNemar test compared exact-span and relaxed-overlap "
        "category success for aligned gold mentions."
    )
else:
    statistical_notes.append(
        "McNemar testing was not performed: " + mcnemar_result["reason"] + "."
    )

report_markdown = f'''# Reproducibility and statistical validation report

## Study configuration

- Dataset: MIMIC-IV discharge summaries
- Reviewed notes: 70
- Gold complication mentions: 491
- Canonical predictions: 546
- Random seed: {RANDOM_SEED}
- Confidence level: {CONFIDENCE_LEVEL:.0%}
- Generated: {datetime.now(timezone.utc).isoformat()}

## Frozen evaluation results

{overall_results_df.to_markdown(index=False)}

## Statistical analysis

{" ".join(statistical_notes)}

## Integrity controls

- Notebook 08 validation passed before packaging.
- Notebook 09 validation passed before packaging.
- Gold-standard annotations were not modified.
- Canonical predictions were not modified.
- Evaluation results were not recomputed.
- Raw patient text was excluded from the public GitHub package.
- SHA256 checksums were calculated for final research assets.

## Reproducibility outputs

- software_versions.csv
- system_environment.csv
- experiment_manifest.json
- project_inventory.csv
- sha256_manifest.csv
- bootstrap_confidence_intervals.csv
- wilson_confidence_intervals.csv
- per_category_confidence_intervals.csv
- mcnemar_test.csv
- effect_size_summary.csv
'''

(REPORTS_DIR / "reproducibility_report.md").write_text(
    report_markdown,
    encoding="utf-8",
)

(REPORTS_DIR / "reproducibility_report.html").write_text(
    "<html><body>" + Markdown(report_markdown).data.replace("\n", "<br>") + "</body></html>",
    encoding="utf-8",
)

print("Research report generated.")

## 24. Reproducibility checklist

In [ ]:
checklist_rows = [
    ("Notebook 08 validation passed", upstream_validation_df.loc[0, "passed"]),
    ("Notebook 09 validation passed", upstream_validation_df.loc[1, "passed"]),
    ("Frozen evaluation summary located", summary_path is not None),
    ("Publication asset manifest located", asset_manifest_path is not None),
    ("Software versions recorded", (MANIFESTS_DIR / "software_versions.csv").exists()),
    ("Experiment manifest generated", (MANIFESTS_DIR / "experiment_manifest.json").exists()),
    ("Project inventory generated", (MANIFESTS_DIR / "project_inventory.csv").exists()),
    ("SHA256 manifest generated", (MANIFESTS_DIR / "sha256_manifest.csv").exists()),
    ("Bootstrap analysis documented", (STATISTICS_DIR / "bootstrap_confidence_intervals.csv").exists()),
    ("Wilson intervals generated", (STATISTICS_DIR / "wilson_confidence_intervals.csv").exists()),
    ("Per-category intervals generated", (STATISTICS_DIR / "per_category_confidence_intervals.csv").exists()),
    ("McNemar applicability documented", (STATISTICS_DIR / "mcnemar_test.csv").exists()),
    ("Dissertation package generated", DISSERTATION_PACKAGE_DIR.exists()),
    ("Poster package generated", POSTER_PACKAGE_DIR.exists()),
    ("Presentation package generated", PRESENTATION_PACKAGE_DIR.exists()),
    ("GitHub README generated", (GITHUB_PACKAGE_DIR / "README.md").exists()),
    ("CITATION.cff generated", (GITHUB_PACKAGE_DIR / "CITATION.cff").exists()),
    ("requirements.txt generated", (GITHUB_PACKAGE_DIR / "requirements.txt").exists()),
    ("environment.yml generated", (GITHUB_PACKAGE_DIR / "environment.yml").exists()),
]

reproducibility_checklist_df = pd.DataFrame(
    checklist_rows,
    columns=["check", "passed"],
)
reproducibility_checklist_df["passed"] = reproducibility_checklist_df["passed"].astype(bool)
reproducibility_checklist_df.to_csv(
    REPORTS_DIR / "reproducibility_checklist.csv",
    index=False,
)
display(reproducibility_checklist_df)

## 25. Create the final ZIP archive

In [ ]:
zip_path = NOTEBOOK_10_DIR / "submission_package.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as archive:
    for path in PACKAGE_DIR.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(PACKAGE_DIR),
            )

print({
    "zip_path": str(zip_path),
    "size_mb": round(zip_path.stat().st_size / (1024 ** 2), 2),
})

## 26. Final validation

The final validation is stored as CSV and JSON. Console output is intentionally concise.

In [ ]:
required_outputs = [
    MANIFESTS_DIR / "software_versions.csv",
    MANIFESTS_DIR / "system_environment.csv",
    MANIFESTS_DIR / "experiment_manifest.json",
    MANIFESTS_DIR / "project_inventory.csv",
    MANIFESTS_DIR / "sha256_manifest.csv",
    STATISTICS_DIR / "bootstrap_confidence_intervals.csv",
    STATISTICS_DIR / "wilson_confidence_intervals.csv",
    STATISTICS_DIR / "per_category_confidence_intervals.csv",
    STATISTICS_DIR / "mcnemar_test.csv",
    STATISTICS_DIR / "effect_size_summary.csv",
    TABLES_DIR / "dissertation_table_overall_performance.csv",
    TABLES_DIR / "figure_numbering_and_captions.csv",
    REPORTS_DIR / "reproducibility_report.md",
    REPORTS_DIR / "reproducibility_checklist.csv",
    GITHUB_PACKAGE_DIR / "README.md",
    GITHUB_PACKAGE_DIR / "LICENSE",
    GITHUB_PACKAGE_DIR / "CITATION.cff",
    GITHUB_PACKAGE_DIR / "requirements.txt",
    GITHUB_PACKAGE_DIR / "environment.yml",
    zip_path,
]

validation_rows = [
    {
        "check": "upstream_validations_passed",
        "passed": bool(upstream_validation_df["passed"].all()),
        "details": "Notebook 08 and Notebook 09",
    },
    {
        "check": "required_outputs_exist",
        "passed": all(path.exists() for path in required_outputs),
        "details": f"{sum(path.exists() for path in required_outputs)}/{len(required_outputs)}",
    },
    {
        "check": "per_category_confidence_intervals_nonempty",
        "passed": (
            (STATISTICS_DIR / "per_category_confidence_intervals.csv").exists()
            and not per_category_ci_df.empty
            and len(per_category_ci_df) == len(category_metrics_df)
        ),
        "details": f"{len(per_category_ci_df)}/{len(category_metrics_df)} categories",
    },
    {
        "check": "submission_zip_nonempty",
        "passed": zip_path.exists() and zip_path.stat().st_size > 0,
        "details": str(zip_path),
    },
    {
        "check": "reproducibility_checklist_passed",
        "passed": bool(reproducibility_checklist_df["passed"].all()),
        "details": f"{int(reproducibility_checklist_df['passed'].sum())}/{len(reproducibility_checklist_df)}",
    },
    {
        "check": "no_raw_clinical_text_intentionally_copied",
        "passed": True,
        "details": "Package copies figures, aggregate tables, reports and documentation only",
    },
]

notebook_10_validation_df = pd.DataFrame(validation_rows)
notebook_10_validation_df.to_csv(
    REPORTS_DIR / "notebook_10_final_validation.csv",
    index=False,
)

validation_report = {
    "notebook": 10,
    "title": "Reproducibility, Statistical Validation and Submission Packaging",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_checks_passed": bool(notebook_10_validation_df["passed"].all()),
    "checks": notebook_10_validation_df.to_dict(orient="records"),
    "statistical_analysis": {
        "bootstrap_status": bootstrap_ci_df["status"].tolist(),
        "mcnemar_status": mcnemar_result["status"],
        "mcnemar_reason": mcnemar_result["reason"],
    },
    "submission_zip": str(zip_path),
}

with (REPORTS_DIR / "notebook_10_report.json").open("w", encoding="utf-8") as handle:
    json.dump(validation_report, handle, indent=2)

display(notebook_10_validation_df)

if not notebook_10_validation_df["passed"].all():
    raise RuntimeError("Notebook 10 validation failed. Review the validation table.")

print("Notebook 10 completed; validation and package files were written successfully.")

## Completion outputs

The main outputs are stored under:

`outputs/notebook_10_reproducibility_statistics_submission/`

The most important files are:

- `submission_package.zip`
- `reports/notebook_10_final_validation.csv`
- `reports/notebook_10_report.json`
- `reports/reproducibility_report.md`
- `manifests/sha256_manifest.csv`
- `manifests/experiment_manifest.json`
- `statistics/bootstrap_confidence_intervals.csv`
- `statistics/wilson_confidence_intervals.csv`
- `statistics/per_category_confidence_intervals.csv`
- `statistics/mcnemar_test.csv`

A statistical test being marked **not applicable** does not constitute a failed notebook. It means the available frozen outputs did not support that test without violating its assumptions.